# 04 - Wav2Vec2 Fine-Tuning
Fine-tune `facebook/wav2vec2-base` for 6-class speech emotion classification on CREMA-D.

This notebook mirrors `src/train.py` and `src/evaluate.py` for interactive experimentation.

In [ ]:
import sys
sys.path.append('..')

import torch
from torch.utils.data import DataLoader
from transformers import Wav2Vec2Processor

from src.config import PRETRAINED_MODEL_NAME, BATCH_SIZE, NUM_EPOCHS, LEARNING_RATE_HEAD, LEARNING_RATE_FINETUNE
from src.dataset import load_crema_d, SpeechEmotionDataset, collate_fn
from src.model import SpeechEmotionModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

In [ ]:
processor = Wav2Vec2Processor.from_pretrained(PRETRAINED_MODEL_NAME)
raw_dataset = load_crema_d()

train_dataset = SpeechEmotionDataset(raw_dataset['train'], processor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [ ]:
model = SpeechEmotionModel().to(device)
criterion = torch.nn.CrossEntropyLoss()

## Phase 1: Train the classifier head (encoder frozen)

In [ ]:
model.freeze_encoder()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE_HEAD)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_values = batch['input_values'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_values)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}')

## Phase 2: Unfreeze and fine-tune end-to-end

In [ ]:
model.unfreeze_encoder()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE_FINETUNE)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_values = batch['input_values'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_values)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f'Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}')

In [ ]:
import os
from src.config import MODEL_SAVE_DIR, MODEL_SAVE_PATH

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
torch.save(model.state_dict(), MODEL_SAVE_PATH)
print('Saved to', MODEL_SAVE_PATH)

## Next steps
- Run `src/evaluate.py` to generate the classification report and confusion matrix.
- Try different learning rates (1e-4, 5e-5, 1e-5) and log results.
- Add audio augmentation (noise, time-stretch, pitch-shift) and compare.